[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/tu-usuario/tu-repo/blob/main/notebooks/Sistema_Recuperacion_Semantica_Imagenes.ipynb)

# 1. Configuración del Entorno y Dependencias

Esta sección detalla la instalación de las dependencias críticas requeridas para el sistema de Recuperación de Imágenes Basada en Contenido (CBIR).

- `open_clip_torch` & `transformers`: Requeridos para la implementación del modelo CLIP (Visión y Lenguaje).
- `faiss-cpu`: Habilita la búsqueda eficiente de similitud y la indexación para la recuperación de información.
- `bitsandbytes` & `accelerate`: Facilitan la cuantización y optimización para el despliegue del Large Language Model (LLM).
- `umap-learn`: Utilizado para la reducción de dimensionalidad avanzada y la visualización de embeddings.

In [ ]:

print("Instalando librerias...")
!pip install open_clip_torch transformers
!pip install faiss-cpu
!pip install bitsandbytes accelerate
!pip install open_clip_torch transformers
!pip install faiss-cpu
!pip install umap-learn --upgrade

# 2. Análisis Exploratorio de Datos (EDA)

Esta fase inicia el pipeline de Análisis Exploratorio de Datos (EDA). Importamos las librerías fundamentales (`pandas`, `numpy`, `matplotlib`, `seaborn`) para la manipulación y visualización de datos.

Se establecen las configuraciones visuales y la semilla aleatoria para garantizar una reproducibilidad absoluta en todos los experimentos. Finalmente, se define el directorio objetivo que contiene el dataset de imágenes.

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter
import cv2
import warnings
import random

# Silenciar warnings no críticos
warnings.filterwarnings("ignore", category=FutureWarning)

# Fijar semilla
random.seed(42)
np.random.seed(42)


# Configuración visual
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8,5)

# Ruta confirmada de imágenes
import os
from pathlib import Path

# Soporte robusto para entornos (Kaggle, Colab, Local)
if os.path.exists("/kaggle/input/tpi-frc-utn-2025-primavera"):
    base_data_path = Path("/kaggle/input/tpi-frc-utn-2025-primavera")
    images_path = str(base_data_path / "dataset/VOC2012_train_val/VOC2012_train_val/JPEGImages")
    annotations_path = str(base_data_path / "dataset/VOC2012_train_val/VOC2012_train_val/Annotations")
    queries_path = str(base_data_path / "questions.csv")
    processed_dir = Path("./")
else:
    # Rutas relativas al repositorio local
    repo_root = Path().resolve()
    if repo_root.name == "notebooks":
        repo_root = repo_root.parent
    
    images_path = str(repo_root / "data" / "raw" / "JPEGImages")
    annotations_path = str(repo_root / "data" / "raw" / "Annotations")
    queries_path = str(repo_root / "data" / "raw" / "questions.csv")
    processed_dir = repo_root / "data" / "processed"
    processed_dir.mkdir(parents=True, exist_ok=True)


Al iterar sobre el dataset, extraemos metadatos esenciales: ancho, alto, relación de aspecto y tamaño del archivo (KB).

Estas métricas se consolidan en un DataFrame estructurado (`eda_df`). Posteriormente, se genera un resumen estadístico para analizar la distribución dimensional y la varianza a lo largo del dataset de imágenes.

In [ ]:

image_files = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
n_images = len(image_files)
print(f"[INFO] Total de imágenes: {n_images}")

# Cargar una muestra para analizar dimensiones
sample_files = np.random.choice(image_files, n_images, replace=False)

widths, heights, aspect_ratios, channels, file_sizes = [], [], [], [], []

for fname in tqdm(sample_files, desc="Analizando imágenes"):
    path = os.path.join(images_path, fname)
    with Image.open(path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)
        aspect_ratios.append(round(w/h, 2))
        channels.append(len(img.getbands()))
    file_sizes.append(os.path.getsize(path) / 1024)  # en KB

# Crear dataframe resumen
eda_df = pd.DataFrame({
    "width": widths,
    "height": heights,
    "aspect_ratio": aspect_ratios,
    "channels": channels,
    "file_size_kb": file_sizes
})

eda_df.describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,10))

sns.histplot(eda_df["width"], bins=30, ax=axes[0,0], color="skyblue")
axes[0,0].set_title("Distribución del ancho")

sns.histplot(eda_df["height"], bins=30, ax=axes[0,1], color="lightgreen")
axes[0,1].set_title("Distribución del alto")

sns.histplot(eda_df["aspect_ratio"], bins=20, ax=axes[1,0], color="salmon")
axes[1,0].set_title("Relación de aspecto (w/h)")

sns.histplot(eda_df["file_size_kb"], bins=30, ax=axes[1,1], color="plum")
axes[1,1].set_title("Tamaño de archivo (KB)")

plt.tight_layout()
plt.show()

Utilizamos la librería `xml.etree.ElementTree` para parsear los archivos de anotación XML que se adhieren al formato Pascal VOC.

Al iterar sobre las anotaciones, extraemos las etiquetas de clase para cada *bounding box*, agregando las frecuencias mediante un `defaultdict`.

La distribución resultante se estructura en un DataFrame y se visualiza a través de un gráfico de barras para proporcionar información sobre el desbalance de clases y la composición del dataset.

In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict

# annotations_path defined globally

class_counts = defaultdict(int)
xml_files = os.listdir(annotations_path)

for xml_file in tqdm(xml_files, desc="Leyendo etiquetas"):
    tree = ET.parse(os.path.join(annotations_path, xml_file))
    root = tree.getroot()
    for obj in root.findall("object"):
        name = obj.find("name").text
        class_counts[name] += 1

# Convertir a dataframe y ordenar
class_df = pd.DataFrame(class_counts.items(), columns=["class", "count"]).sort_values(by="count", ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x="count", y="class", data=class_df, palette="viridis")
plt.title("Distribución de clases en Pascal VOC 2012")
plt.xlabel("Cantidad de imágenes")
plt.ylabel("Clase")
plt.show()

# 3. Análisis de Clustering No Supervisado

Esta sección detalla el clustering no supervisado del dataset de imágenes, aprovechando los embeddings de CLIP para agrupar imágenes basándose en su similitud semántica.

1. **Generación de Embeddings**: Inicializa el modelo CLIP (`ViT-B-32`) para calcular representaciones de características para cada imagen, capturando el contexto semántico profundo.
2. **Reducción de Dimensionalidad (UMAP/t-SNE)**: Aplica UMAP (o t-SNE como alternativa) para proyectar los embeddings de alta dimensionalidad en un espacio 2D para su visualización.
3. **Clustering K-Means**: Ejecuta el algoritmo K-Means a través de varios valores de K (número de clusters) sobre los embeddings originales de alta dimensionalidad para formar grupos semánticos.
4. **Visualización y Evaluación**: Renderiza los resultados del clustering en 2D y calcula métricas de evaluación intrínseca para evaluar la cohesión y separación de los clusters.

In [ ]:
# ---- Configuración ----
BATCH_SIZE = 64
CLIP_MODEL = "ViT-B-32"
RANDOM_SEED = 42
N_COMPONENTS_UMAP = 2
K_VALUES = [10, 20, 40] 
SAVE_EMBEDDINGS = True   
EMBEDDINGS_PATH = "clip_embeddings_full.npy"


import os
from pathlib import Path

# Soporte robusto para entornos (Kaggle, Colab, Local)
if os.path.exists("/kaggle/input/tpi-frc-utn-2025-primavera"):
    base_data_path = Path("/kaggle/input/tpi-frc-utn-2025-primavera")
    images_path = str(base_data_path / "dataset/VOC2012_train_val/VOC2012_train_val/JPEGImages")
    annotations_path = str(base_data_path / "dataset/VOC2012_train_val/VOC2012_train_val/Annotations")
    queries_path = str(base_data_path / "questions.csv")
    processed_dir = Path("./")
else:
    # Rutas relativas al repositorio local
    repo_root = Path().resolve()
    if repo_root.name == "notebooks":
        repo_root = repo_root.parent
    
    images_path = str(repo_root / "data" / "raw" / "JPEGImages")
    annotations_path = str(repo_root / "data" / "raw" / "Annotations")
    queries_path = str(repo_root / "data" / "raw" / "questions.csv")
    processed_dir = repo_root / "data" / "processed"
    processed_dir.mkdir(parents=True, exist_ok=True)


# ---- Imports ----
import os, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image
import torch
from sklearn.cluster import KMeans
import umap
import open_clip
import warnings
warnings.filterwarnings("ignore")

# ---- Reproducibilidad ----
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ---- Verificación de carpeta ----
if not os.path.exists(images_path):
    raise FileNotFoundError(f"No se encontró la carpeta de imágenes: {images_path}")

all_images = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
n_total = len(all_images)
print(f"Total imágenes disponibles: {n_total}")

# ---- Cargar modelo CLIP ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando dispositivo:", device)
model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained='openai')
model.to(device)
model.eval()

# ---- Función: generar embeddings ----
def images_to_embeddings(file_list, batch_size=BATCH_SIZE, device=device):
    embeddings = []
    n = len(file_list)
    for i in tqdm(range(0, n, batch_size), desc="Calculando embeddings"):
        batch_files = file_list[i:i+batch_size]
        imgs = []
        for fname in batch_files:
            img = Image.open(os.path.join(images_path, fname)).convert("RGB")
            imgs.append(preprocess(img).unsqueeze(0))
        batch_tensor = torch.cat(imgs, dim=0).to(device)
        with torch.no_grad():
            emb = model.encode_image(batch_tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            embeddings.append(emb.cpu().numpy())
        del batch_tensor, emb
        torch.cuda.empty_cache()
    embeddings = np.vstack(embeddings)
    return embeddings

# ---- Cargar o calcular embeddings ----
if os.path.exists(EMBEDDINGS_PATH):
    print(f"Embeddings ya existen: cargando desde {EMBEDDINGS_PATH}")
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    print("Calculando embeddings para todo el dataset (puede tardar unos minutos)...")
    embeddings = images_to_embeddings(all_images, batch_size=BATCH_SIZE)
    if SAVE_EMBEDDINGS:
        np.save(EMBEDDINGS_PATH, embeddings)
        print(f"Embeddings guardados en {EMBEDDINGS_PATH}")

print("Embeddings shape:", embeddings.shape)

# ---- SOLUCIÓN: Reducir a 2D con UMAP (VERSIÓN CORREGIDA) ----
print("Aplicando reducción de dimensionalidad...")

# Opción 1: Intentar UMAP con manejo de errores
try:
    reducer = umap.UMAP(
        n_components=N_COMPONENTS_UMAP,
        n_neighbors=15,
        min_dist=0.1,  
        metric='cosine',
        random_state=RANDOM_SEED,
        low_memory=False 
    )
    emb_2d = reducer.fit_transform(embeddings)
    print("✓ UMAP completado exitosamente")
    
except Exception as e:
    print(f"✗ UMAP falló: {e}")
    print("Usando t-SNE como alternativa...")
    
    # Opción 2: t-SNE como alternativa
    from sklearn.manifold import TSNE
    emb_2d = TSNE(
        n_components=N_COMPONENTS_UMAP,
        random_state=RANDOM_SEED,
        perplexity=30,
        learning_rate='auto',
        init='random'
    ).fit_transform(embeddings)
    print("✓ t-SNE completado exitosamente")

print("Embeddings 2D shape:", emb_2d.shape)

# ---- Visualización comparativa para distintos K ----
fig, axes = plt.subplots(1, len(K_VALUES), figsize=(18,6))
for idx, K in enumerate(K_VALUES):
    print(f"\nAplicando KMeans con k={K}...")
    kmeans = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    df_viz = pd.DataFrame({
        "x": emb_2d[:,0],
        "y": emb_2d[:,1],
        "cluster": cluster_labels
    })
    sns.scatterplot(
        data=df_viz, x="x", y="y", hue="cluster",
        palette=sns.color_palette("tab20", n_colors=K),
        s=10, alpha=0.7, linewidth=0, ax=axes[idx]
    )
    axes[idx].set_title(f"KMeans (k={K})", fontsize=12)
    axes[idx].axis("off")
    axes[idx].legend([], [], frameon=False)

plt.suptitle("Comparación de Clustering sobre Embeddings CLIP + Reducción Dimensional", fontsize=14)
plt.tight_layout()
plt.show()

# ---- Guardar dataframe del último clustering ----
df_viz["file"] = all_images
df_viz.to_csv("umap_kmeans_full.csv", index=False)
print("Archivo umap_kmeans_full.csv guardado con coordenadas y clusters finales.")

# ---- Métricas de evaluación ----
from sklearn.metrics import silhouette_score, calinski_harabasz_score

print("\n" + "="*50)
print("EVALUACIÓN DE CLUSTERING")
print("="*50)

for K in K_VALUES:
    kmeans = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    
    sil_score = silhouette_score(embeddings, cluster_labels, sample_size=5000, random_state=42)
    ch_score = calinski_harabasz_score(embeddings, cluster_labels)
    
    print(f"K={K}:")
    print(f"  - Silhouette Score: {sil_score:.4f}")
    print(f"  - Calinski-Harabasz Score: {ch_score:.4f}")
    print()

La siguiente implementación evalúa la calidad numérica de los clusters de K-Means generados a partir de los embeddings de CLIP, analizando el rendimiento a través de diferentes valores de K (10, 20 y 40).

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, homogeneity_score

print("Evaluando calidad de los clusters...\n")

# --- Cargar embeddings y resultados previos ---
if 'embeddings' not in locals():
    print("Cargando embeddings desde archivo...")
    embeddings = np.load("clip_embeddings_full.npy")

if 'emb_2d' not in locals():
    print("Cargando proyección UMAP desde archivo CSV...")
    df_viz = pd.read_csv("umap_kmeans_full.csv")
    emb_2d = df_viz[['x','y']].values
else:
    df_viz = pd.read_csv("umap_kmeans_full.csv")

# --- Evaluar distintos K ---
metrics = []
for K in [10, 20, 40]:
    print(f"Reajustando KMeans (k={K})...")
    kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)

    sil = silhouette_score(embeddings, labels, sample_size=5000, random_state=42)
    ch  = calinski_harabasz_score(embeddings, labels)
    db  = davies_bouldin_score(embeddings, labels)

    metrics.append({
        "k": K,
        "Silhouette": sil,
        "Calinski-Harabasz": ch,
        "Davies-Bouldin": db
    })

# --- Mostrar tabla de resultados ---
df_metrics = pd.DataFrame(metrics)
display(df_metrics.style.background_gradient(cmap="YlGnBu", axis=None))

Esta sección ejecuta una **evaluación cuantitativa supervisada** del rendimiento del clustering de K-Means. El objetivo principal es medir la alineación entre los clusters no supervisados y las verdaderas etiquetas de clase (ground-truth) del dataset Pascal VOC.

In [ ]:
import xml.etree.ElementTree as ET
from sklearn.metrics import adjusted_rand_score, homogeneity_score
from collections import defaultdict, Counter

# annotations_path defined globally

# --- 1. Asociar cada imagen con una clase principal ---
def parse_class_per_image(annotations_path):
    label_map = {}
    xml_files = [f for f in os.listdir(annotations_path) if f.endswith(".xml")]

    for xml_file in tqdm(xml_files, desc="Leyendo etiquetas XML"):
        image_id = xml_file.replace(".xml", ".jpg")
        tree = ET.parse(os.path.join(annotations_path, xml_file))
        root = tree.getroot()
        classes = [obj.find("name").text for obj in root.findall("object")]
        if classes:
            # tomar la clase más frecuente en esa imagen
            main_class = Counter(classes).most_common(1)[0][0]
            label_map[image_id] = main_class
    return label_map

label_map = parse_class_per_image(annotations_path)
print(f"Total de imágenes con etiquetas: {len(label_map)}")

# --- 2. Alinear etiquetas con el orden de los embeddings ---
labels_true = []
valid_images = []
for img in all_images:
    if img in label_map:
        labels_true.append(label_map[img])
        valid_images.append(img)

labels_true = np.array(labels_true)
print(f"Etiquetas reales obtenidas para {len(labels_true)} imágenes.")

# --- 3. Convertir nombres de clase a números (para métricas sklearn) ---
unique_classes = sorted(list(set(labels_true)))
class_to_id = {cls: i for i, cls in enumerate(unique_classes)}
labels_true_ids = np.array([class_to_id[c] for c in labels_true])

# --- 4. Cargar embeddings y proyección si no están ---
if 'embeddings' not in locals():
    embeddings = np.load("clip_embeddings_full.npy")

# --- 5. Recalcular clustering con K=20 (similar a las 20 clases reales) ---
K = 20
print(f"\nAplicando KMeans con k={K} (comparación con clases reales)...")
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings[:len(labels_true)])

# --- 6. Calcular métricas supervisadas ---
ari = adjusted_rand_score(labels_true_ids, cluster_labels)
homo = homogeneity_score(labels_true_ids, cluster_labels)

print(f"\n[METRICS] Resultados de comparación con clases reales:")
print(f"Adjusted Rand Index (ARI): {ari:.3f}")
print(f"Homogeneity Score: {homo:.3f}")

# --- 7. Tabla resumen (por clase y cluster) ---
df_eval = pd.DataFrame({
    "file": valid_images,
    "class": labels_true,
    "cluster": cluster_labels
})

cross_tab = pd.crosstab(df_eval['class'], df_eval['cluster'])
plt.figure(figsize=(12,6))
sns.heatmap(cross_tab, cmap="YlGnBu", cbar=True)
plt.title("Matriz clase vs cluster (frecuencia de asignación)")
plt.xlabel("Cluster KMeans")
plt.ylabel("Clase real")
plt.show()

A continuación, construimos y visualizamos una **matriz de co-ocurrencia**. Este análisis destaca la frecuencia con la que múltiples clases distintas aparecen simultáneamente dentro de la misma imagen, proporcionando información sobre las relaciones entre clases y las dependencias contextuales.

In [ ]:
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

xml_files = [f for f in os.listdir(annotations_path) if f.endswith(".xml")]

cooc = Counter()
for xml_file in xml_files:
    tree = ET.parse(os.path.join(annotations_path, xml_file))
    objs = [obj.find("name").text for obj in tree.findall("object")]
    for c1, c2 in itertools.combinations(sorted(set(objs)), 2):
        cooc[(c1, c2)] += 1

cooc_df = pd.DataFrame([
    {"class1": c1, "class2": c2, "count": v} for (c1, c2), v in cooc.items()
])
pivot = cooc_df.pivot(index="class1", columns="class2", values="count").fillna(0)
plt.figure(figsize=(10,8))
sns.heatmap(pivot + pivot.T, cmap="YlGnBu", cbar=True)
plt.title("Matriz de co-ocurrencia de clases (Pascal VOC 2012)")
plt.show()

In [ ]:
# --- FUNCIÓN AUXILIAR: Mover Modelo CLIP ---
def set_clip_device(target_device: str):
    """Mueve el modelo CLIP a la CPU o GPU para gestionar la VRAM."""
    global model 
    
    if target_device == "cpu":
        if next(model.parameters()).device.type == 'cuda':
            model.to('cpu')
            torch.cuda.empty_cache() # Liberar VRAM
            # print("CLIP movido a CPU.")
    elif target_device == "cuda":
        if next(model.parameters()).device.type == 'cpu':
            model.to('cuda')
            # print("CLIP movido a GPU.")

# 4. Inicialización de la Base de Datos Vectorial (FAISS)

Este módulo inicializa y construye un índice FAISS (Facebook AI Similarity Search) utilizando los embeddings precalculados de CLIP, permitiendo una búsqueda de similitud altamente escalable y eficiente.

- **Ingesta de Embeddings**: Carga los embeddings de CLIP (`clip_embeddings_full.npy`) y los convierte a `float32`, cumpliendo con los requisitos de FAISS.
- **Instanciación del Índice**: Instancia `faiss.IndexFlatIP`. Este índice se basa en el Producto Interno (Inner Product), el cual es matemáticamente equivalente a la similitud del coseno cuando los vectores están normalizados en L2.
- **Indexación y Persistencia**: Ingesta todos los embeddings normalizados en la estructura del índice. El índice resultante se serializa y guarda en disco (`faiss_index.bin`) para garantizar la reproducibilidad y minimizar la carga computacional futura.

In [ ]:
import numpy as np
import faiss
import torch
import open_clip

# ---- Configuración ----
RANDOM_SEED = 42
CLIP_MODEL = "ViT-B-32"
EMBEDDINGS_PATH = "clip_embeddings_full.npy"
INDEX_PATH = "faiss_index.bin"
K = 10

# ---- Cargar Embeddings ----
print(f"Cargando embeddings desde {EMBEDDINGS_PATH}...")
embeddings = np.load(EMBEDDINGS_PATH).astype('float32')
D = embeddings.shape[1] # Dimensionalidad (debería ser 512 para ViT-B/32)

# 1. Inicializar el índice FAISS
# Usamos IndexFlatIP para Producto Interno (equivalente a Coseno con vectores normalizados)
index = faiss.IndexFlatIP(D)

# 2. Asegurarse de que el índice esté vacío y añadir los vectores
print(f"Añadiendo {embeddings.shape[0]} vectores de {D} dimensiones al índice FAISS...")
index.add(embeddings)

# 3. Guardar el índice a disco (requerido para entregables)
faiss.write_index(index, INDEX_PATH)
print(f"[SUCCESS] Índice FAISS creado y guardado en {INDEX_PATH}.")
print(f"Total de vectores en el índice: {index.ntotal}")

Esta celda implementa la lógica central para la Recuperación de Imágenes Basada en Contenido (CBIR).

- **Inicialización de Componentes**: Instancia el tokenizador y el modelo CLIP para la codificación de texto, y carga el inventario de metadatos de las imágenes.
- **Función `search_baseline`**: Define el pipeline para proyectar una consulta de texto en el espacio de embeddings de CLIP.
- **Recuperación Eficiente**: Consulta el índice FAISS para recuperar los top-K embeddings de imágenes más similares semánticamente a través de la similitud del coseno.
- **Mapeo de Resultados**: Mapea los índices de FAISS y los scores de similitud a sus correspondientes identificadores de imagen para el procesamiento y visualización posteriores.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained='openai')
model.to(device)
model.eval()
tokenizer = open_clip.get_tokenizer(CLIP_MODEL)

# Cargar el inventario (necesario para mapear IDs)
df_inventory = pd.read_csv("umap_kmeans_full.csv") # O el archivo generado en 1.1

# CORRECCIÓN CRÍTICA: Asegurar que el índice de df_inventory esté limpio y alineado (0, 1, 2, ...)
df_inventory.reset_index(drop=True, inplace=True) 

# La K debe definirse antes de usarse
K = 10 #Ver cual poner

def search_baseline(query_text: str, k: int = K) -> pd.DataFrame:
    """
    Realiza la búsqueda baseline: codifica el texto con CLIP y consulta FAISS.
    Asume que la consulta ya está en el idioma de entrenamiento (inglés).
    """
    # 1. Codificar la consulta de texto a embedding
    with torch.no_grad():
        text_token = tokenizer(query_text).to(device)
        text_emb = model.encode_text(text_token)
        # Normalizar el embedding
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
        query_vector = text_emb.cpu().numpy().astype('float32')

    # 2. Consultar el índice FAISS
    D, I = index.search(query_vector, k)

    # 3. Recuperar y mapear los resultados
    results = df_inventory.iloc[I[0]].copy()
    
    # Asignar la distancia/similitud (score) y la consulta
    results['similarity_score'] = D[0]
    results['query'] = query_text
    
    if 'image_id' not in results.columns:
        # Si no existe, la creamos (tomando el nombre de archivo sin la extensión y reemplazando '_')
        results['image_id'] = results['file'].apply(lambda x: os.path.splitext(x)[0])

    return results[['image_id', 'similarity_score', 'file', 'query']]

# ---- Ejemplo de prueba ----
print("\n--- TEST DE BÚSQUEDA BASELINE (en inglés) ---")
test_query = "a photo of a yellow car on a street"
baseline_results = search_baseline(test_query)

print(f"Resultados para: '{test_query}'")
display(baseline_results.head(K))


# 5. Integración de Modelos de Lenguaje Grande (LLM)

Esta sección es crítica, ya que despliega el Large Language Model (`Phi-3-mini-4k-instruct`) y define su función dentro de la arquitectura de recuperación avanzada.

- **Cuantización e Inicialización**: Carga el LLM utilizando un esquema de cuantización de 4 bits a través de `BitsAndBytesConfig`. Esto optimiza significativamente el consumo de VRAM, permitiendo la ejecución en entornos de hardware limitados.
- **Pipeline de Generación**: Configura un pipeline optimizado de generación de texto, aprovechando el mapeo automático de dispositivos (`device_map="auto"`).
- **Función `reformulate_for_negation`**: Implementa la lógica de *prompting* que guía al LLM para realizar tres tareas principales: traducción de la consulta, expansión semántica (vía sinónimos) y detección/aislamiento de negaciones (por ejemplo, aislando objetos que deben ser excluidos). El LLM produce esta información estructurada en un formato JSON estricto, lo cual es esencial para la fase posterior de reranking.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch
import json
import os # Necesario para las rutas

# ---- Configuración del modelo ----
LLM_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
K = 10 

print(f"\n--- CARGANDO LLM: {LLM_MODEL_NAME} ---")

# Configuración de cuantización (según la advertencia de `torch_dtype` y `load_in_4bit`)
# Esto es más limpio y el método preferido por Hugging Face.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

try:
    # 1. Carga del modelo con configuración explícita de cuantización (resuelve advertencias)
    model_llm = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        quantization_config=bnb_config, # Usa la configuración BNB
        device_map="auto",             # Deja que accelerate decida dónde cargar (generalmente GPU:0)
        torch_dtype=torch.bfloat16,     # Usa dtype en lugar de torch_dtype (resuelve advertencia)
        # Aquí ya no se usa load_in_4bit ni torch_dtype directamente.
    )
    tokenizer_llm = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
    
    # 2. Crear el pipeline SIN especificar 'device', ya que el modelo ya está en la GPU.
    llm_pipeline = pipeline(
        "text-generation",
        model=model_llm,
        tokenizer=tokenizer_llm,
        # ! IMPORTANTE: Eliminamos 'device=0' o 'device=-1' !
    )
    print("[SUCCESS] LLM cargado y listo para inferencia (usando Cuantización de 4 bits).")

except Exception as e:
    # Si esta carga falla, es OOM o un problema de dependencia.
    print(f"[ERROR] Error CRÍTICO al cargar el LLM: {e}. El Reranking no funcionará.")
    llm_pipeline = None

# ---- FUNCIÓN CRÍTICA: REFORMULACIÓN Y NEGACIÓN (Implementa Punto 1.1 y 3.1) ----
# (Esta función permanece igual, ya que solo llama al llm_pipeline)
# ... [Definición de reformulate_for_negation] ...
def reformulate_for_negation(query_es: str):
    """
    Usa el LLM para traducir/reformular, aislar negaciones y devolver un JSON estructurado.
    """
    if llm_pipeline is None:
        # Fallback simple si el LLM no se pudo cargar
        pos_q_fallback = query_es.replace("no", "").replace("sin", "").strip()
        return pos_q_fallback, None, False

    system_prompt = (
        "You are an expert query analysis and reformulation system for CLIP image search. "
        "The user provides a query in Spanish. Your task is to: "
        "1. Translate the query to English and enrich it with 1-2 synonyms (POSITIVE_QUERY). "
        "2. If a negation (e.g., 'no', 'sin') is found, identify the general object and the negated attribute/concept (EXCLUSION_QUERY). "
        "3. Output the result in a clean, strict JSON format with exactly three fields: "
        "POSITIVE_QUERY (string), EXCLUSION_QUERY (string, or null if no negation), and HAS_NEGATION (boolean). "
        "The response must ONLY contain the JSON object."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Query in Spanish: \"{query_es}\""}
    ]
    
    prompt = llm_pipeline.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    try:
        response = llm_pipeline(
            prompt,
            max_new_tokens=150,
            do_sample=False, 
            temperature=0.01,
            return_full_text=False
        )
        json_output = response[0]['generated_text'].strip()
        
        if json_output.startswith("```"):
            json_output = json_output.split("```")[1].replace("json", "").strip()

        analysis = json.loads(json_output)
        
        pos_query = analysis.get('POSITIVE_QUERY', query_es)
        excl_query = analysis.get('EXCLUSION_QUERY', None)
        has_neg = analysis.get('HAS_NEGATION', False)
        
        return pos_query, excl_query, has_neg
        
    except Exception as e:
        print(f"[ERROR] Fallo de LLM: {e}. Usando POSITIVE_QUERY simple como fallback.")
        pos_q_fallback = query_es.replace("no", "").replace("sin", "").strip()
        excl_q_fallback = query_es if "no" in query_es or "sin" in query_es else None
        return pos_q_fallback, excl_q_fallback, True if excl_q_fallback else False

In [ ]:
# ===============================================
# MOSTRADOR DE IMAGENES
# ===============================================

from PIL import Image
import matplotlib.pyplot as plt
import os
import textwrap
import pandas as pd

# Ruta base de las imágenes (usamos la misma que ya tienes definida)
import os
from pathlib import Path

# Soporte robusto para entornos (Kaggle, Colab, Local)
if os.path.exists("/kaggle/input/tpi-frc-utn-2025-primavera"):
    base_data_path = Path("/kaggle/input/tpi-frc-utn-2025-primavera")
    images_path = str(base_data_path / "dataset/VOC2012_train_val/VOC2012_train_val/JPEGImages")
    annotations_path = str(base_data_path / "dataset/VOC2012_train_val/VOC2012_train_val/Annotations")
    queries_path = str(base_data_path / "questions.csv")
    processed_dir = Path("./")
else:
    # Rutas relativas al repositorio local
    repo_root = Path().resolve()
    if repo_root.name == "notebooks":
        repo_root = repo_root.parent
    
    images_path = str(repo_root / "data" / "raw" / "JPEGImages")
    annotations_path = str(repo_root / "data" / "raw" / "Annotations")
    queries_path = str(repo_root / "data" / "raw" / "questions.csv")
    processed_dir = repo_root / "data" / "processed"
    processed_dir.mkdir(parents=True, exist_ok=True)


def display_results(df_results: pd.DataFrame, title: str):
    """Muestra las imágenes de los resultados en una cuadrícula 2x5."""
    
    wrapped_title = '\n'.join(textwrap.wrap(title, 60))
    
    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    axes = axes.flatten()
    
    fig.suptitle(wrapped_title, fontsize=14, y=1.02)

    # CORRECCIÓN: Usar zip para iterar sobre los ejes y las filas del DF.
    # df_results.iterrows() devuelve (índice_original, fila),
    # mientras que zip(axes, df_results.iterrows()) asegura que usamos
    # el eje correcto para cada fila.
    for ax, (idx, row) in zip(axes, df_results.iterrows()):
        try:
            # Reconstruir el nombre de archivo a partir del 'file'
            img_filename = row['file']
            img_path = os.path.join(images_path, img_filename)
            
            img = Image.open(img_path).convert("RGB")
            
            # Mostrar la imagen y el score
            ax.imshow(img)
            # El score puede ser 'similarity_score' (baseline) o 'reranked_score'
            score_col = 'similarity_score' if 'similarity_score' in row else 'reranked_score'
            ax.set_title(f"Score: {row[score_col]:.3f}", fontsize=10)
            ax.axis('off')
        
        except FileNotFoundError:
            ax.text(0.5, 0.5, "[ERROR] Image Not Found", ha='center', va='center')
            ax.axis('off')
        
        except Exception as e:
            # Usar ax en lugar de axes[i]
            ax.text(0.5, 0.5, f"Error: {e}", ha='center', va='center')
            ax.axis('off')
            
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


En la siguiente celda, evaluamos dos estrategias distintas de consulta para el sistema de recuperación de imágenes, destacando las mejoras de rendimiento logradas al integrar el LLM en el pipeline de preprocesamiento de consultas.

1. **Búsqueda Baseline (Consulta Directa)**:
Ejecuta una búsqueda directa utilizando una traducción sencilla de la consulta en español (ej. "a person riding a bicycle on the street"), basándose estrictamente en el pipeline estándar de CLIP + FAISS.

2. **Búsqueda Reformulada (Mejorada por LLM)**:
Utiliza la función `reformulate_for_negation`, permitiendo que el LLM `Phi-3` procese y enriquezca la consulta original en español. El LLM genera una consulta semánticamente enriquecida en inglés (ej. inyectando sinónimos contextuales), la cual luego se pasa al pipeline de CLIP + FAISS.

El objetivo es llevar a cabo un análisis comparativo (tanto visual como cuantitativo) para determinar si la expansión de la consulta impulsada por el LLM produce una mayor relevancia en la recuperación y mejores scores de similitud entre los resultados principales.

In [ ]:
# Consulta que el usuario ingresaría (en español)
USER_QUERY_ES = "animal grande que vuela" 

# Consulta para el BASELINE (Traducción directa o simple, sin el LLM ni expansión)
# Usaremos una traducción simple para simular el caso "sin LLM"
BASELINE_QUERY_EN = "a large flying animal" 

# ---- Paso 2: Búsqueda BASELINE (Solo CLIP y FAISS) ----
print("\n--- BÚSQUEDA 1: BASELINE (Consulta Simple en Inglés) ---")
baseline_results = search_baseline(BASELINE_QUERY_EN)

print(f"[SUCCESS] Búsqueda Baseline (K={K}) para: '{BASELINE_QUERY_EN}'")
display(baseline_results.head(K))

# Visualización Baseline
display_results(
    baseline_results,
    f"BASELINE (Sin LLM) - Consulta: '{BASELINE_QUERY_EN}'"
)

# ---- Paso 3: Búsqueda con REFORMULACIÓN (LLM: Traducción + Expansión) ----

# Generar la consulta reformulada usando el LLM (usa la función definida en la celda 2.3)
print("\n--- BÚSQUEDA 2: REFORMULACIÓN (LLM: Traducción y Expansión) ---")

# CORRECCIÓN CLAVE: Desempaquetar los tres valores que devuelve la función
pos_q_en, excl_q_en, has_negation = reformulate_for_negation(USER_QUERY_ES) 

# La consulta reformulada para esta prueba es la Positiva (pos_q_en)
reformulated_query_en = pos_q_en
    
print(f"Consulta Original (ES): '{USER_QUERY_ES}'")
print(f"Consulta Reformulada (EN): '{reformulated_query_en}'")

# Ejecutar la búsqueda con la consulta reformulada
reformulated_results = search_baseline(reformulated_query_en)

print(f"[SUCCESS] Búsqueda Reformulada (K={K}) para: '{reformulated_query_en}'")
display(reformulated_results.head(K))

# Visualización Reformulación
display_results(
    reformulated_results,
    f"REFORMULACIÓN (Con LLM) - Consulta: '{reformulated_query_en}'"
)

# 6. Pipeline de Reranking Avanzado

Este módulo define la lógica de búsqueda avanzada requerida para manejar eficazmente consultas que contienen negaciones (ej. "coches sin gente").

- **`encode_query`**: Una función de utilidad que proyecta cualquier consulta de texto en un vector de embedding normalizado dentro del espacio latente de CLIP.
- **`search_with_reranking`**: Implementa el mecanismo de *reranking*. Recupera un conjunto inicial de candidatos y penaliza las imágenes que exhiben alta similitud con el vector de exclusión aislado (generado por el LLM). La penalización se escala mediante un factor predefinido `PENALTY_WEIGHT` (por defecto: 0.5).
- **`tpi_search` (Función Maestra)**: Orquesta el flujo de recuperación de extremo a extremo. Consulta al LLM para la reformulación de la consulta y la detección de negaciones, para luego rutear dinámicamente la ejecución hacia la búsqueda baseline (si no se detecta negación) o hacia el pipeline de reranking penalizado (si existen exclusiones).

In [ ]:
# --- CONFIGURACIÓN DE BÚSQUEDA ---
KAGGLE_INPUT_DIR = "/kaggle/input/tpi-frc-utn-2025-primavera"
QUERIES_PATH = os.path.join(KAGGLE_INPUT_DIR, "questions.csv")
SUBMISSION_PATH = "submission.csv"
K = 10
PENALTY_WEIGHT = 0.5


def encode_query(query_text: str):
    """
    Codifica una consulta de texto a un embedding normalizado con CLIP.
    Usa las variables globales 'model' y 'tokenizer' cargadas en la Fase 1.
    """
    global model, tokenizer 
    # Usamos 'device' que ya se cargó en la Fase 1.
    device = next(model.parameters()).device 
    
    with torch.no_grad():
        text_token = tokenizer(query_text).to(device)
        text_emb = model.encode_text(text_token)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
        return text_emb.cpu().numpy().astype('float32').squeeze()


def search_with_reranking(positive_query: str, exclusion_query: str, 
                          penalty_weight: float = PENALTY_WEIGHT, k: int = K) -> pd.DataFrame:
    """Implementa el reranking penalizando el término de exclusión."""
    global index, embeddings, df_inventory
    
    if not exclusion_query or exclusion_query.strip() == "":
        # Fallback si la exclusión es inválida
        print(f"⚠️ Reranking Fallback: Excl query ('{exclusion_query}') inválida. Usando Baseline.")
        return search_baseline(positive_query, k)

    # 1. Búsqueda Inicial con la Consulta Positiva
    v_pos = encode_query(positive_query)
    N_CANDIDATES = k * 5 
    
    D_pos, I_candidates = index.search(v_pos.reshape(1, -1), N_CANDIDATES)
    v_candidates = embeddings[I_candidates[0]]
    scores_original = D_pos[0]
    
    # 2. Vector de Exclusión y Similitudes de Penalización
    v_excl = encode_query(exclusion_query)
    scores_exclusion = np.dot(v_candidates, v_excl.T)
    
    # 3. Aplicar el Reranking (Penalización)
    scores_reranked = scores_original - penalty_weight * scores_exclusion
    
    # 4. Reordenar y obtener Top-K
    sorted_indices = np.argsort(scores_reranked)[::-1]
    top_k_indices = I_candidates[0][sorted_indices[:k]]
    
    # 5. Mapear y devolver
    results = df_inventory.iloc[top_k_indices].copy()
    # Aseguramos que las columnas de scores y ID estén presentes y correctas
    results['image_id'] = results['image_id']
    results['reranked_score'] = scores_reranked[sorted_indices[:k]]
    results['original_score'] = scores_original[sorted_indices[:k]]
    results['exclusion_score'] = scores_exclusion[sorted_indices[:k]]
    
    return results[['image_id', 'reranked_score', 'original_score', 'exclusion_score', 'file']]


# --- FUNCIÓN MAESTRA (Flujo del TPI) ---
def tpi_search(query_es: str, penalty_weight: float = PENALTY_WEIGHT, k: int = K):
    """Ejecuta el flujo completo: LLM (reformulación) + FAISS (+ Reranking)."""
    global llm_pipeline # Aseguramos el acceso al LLM cargado
    
    # 1. Reformulación con LLM
    # Nota: la función reformulate_for_negation debe estar definida y ejecutada previamente.
    pos_q_en, excl_q_en, has_negation = reformulate_for_negation(query_es)

    # 2. Búsqueda y Reranking
    if has_negation and excl_q_en:
        results = search_with_reranking(pos_q_en, excl_q_en, penalty_weight, k)
        results['method'] = 'Reranking'
        results['final_score'] = results['reranked_score'] 
    else:
        results = search_baseline(pos_q_en, k)
        results['method'] = 'Reformulación'
        results['final_score'] = results['similarity_score']

    return results

# 7. Generación de Salida (Submission)

Esta celda final compila los resultados de la evaluación y genera el archivo `submission.csv`. Para un análisis cualitativo riguroso, particularmente en consultas complejas, también renderiza las imágenes recuperadas junto con sus scores finales de similitud correspondientes.

In [ ]:
print("Preparando df_inventory para submission...")


if 'df_inventory' not in locals():
    # Cargar desde archivo si no está en memoria
    df_inventory = pd.read_csv("umap_kmeans_full.csv")

# Crear columna image_id si no existe
if 'image_id' not in df_inventory.columns:
    print("Creando columna 'image_id' en df_inventory...")
    df_inventory['image_id'] = df_inventory['file'].apply(
        lambda x: os.path.splitext(x)[0]
    )
    print(f"[SUCCESS] Columna 'image_id' creada. Ejemplo: {df_inventory['image_id'].iloc[0]}")

# Resetear índice para asegurar alineación
df_inventory.reset_index(drop=True, inplace=True)
print(f"df_inventory shape: {df_inventory.shape}")

# --- Cargar Queries ---
try:
    df_queries = pd.read_csv(QUERIES_PATH)
    df_queries.rename(columns={'query': 'query_es'}, inplace=True)
    queries_es_list = df_queries.set_index('qid')['query_es'].to_dict()
    print(f"[SUCCESS] Queries cargadas. Total: {len(queries_es_list)}")
    print(f"Primeras 5 queries: {list(queries_es_list.items())[:5]}")
except Exception as e:
    print(f"[ERROR] ERROR: No se pudo cargar questions.csv. {e}")
    queries_es_list = {}


def safe_search_with_display(query_es: str, qid: str, k: int = K):
    """
    Función segura que maneja errores en queries complejas y muestra resultados.
    """
    try:
        # Intentar el flujo completo con LLM
        if 'llm_pipeline' in globals() and llm_pipeline is not None:
            # Usar la función de reformulación para negaciones
            pos_q_en, excl_q_en, has_negation = reformulate_for_negation(query_es)
            
            print(f"  🔍 Traducción/Reformulación:")
            print(f"     POSITIVA: '{pos_q_en}'")
            if has_negation and excl_q_en:
                print(f"     EXCLUSIÓN: '{excl_q_en}'")
                print(f"  🔄 Usando reranking para negación")
                results = search_with_reranking(pos_q_en, excl_q_en, PENALTY_WEIGHT, k)
                method = "Reranking (con negación)"
            else:
                print(f"  🔄 Usando reformulación simple")
                results = search_baseline(pos_q_en, k)
                method = "Reformulación"
        else:
            # Fallback: búsqueda baseline directa
            print(f"  ⚠️ LLM no disponible, usando baseline directa")
            results = search_baseline(query_es, k)
            method = "Baseline (sin LLM)"
            pos_q_en = query_es
            
        return results, method, pos_q_en, has_negation
        
    except Exception as e:
        print(f"  [ERROR] Error en procesamiento: {e}")
        print(f"  🆘 Usando fallback baseline...")
        # Fallback extremo: búsqueda baseline con la query original
        try:
            results = search_baseline(query_es, k)
            method = "Fallback Baseline"
            pos_q_en = query_es
            return results, method, pos_q_en, False
        except:
            # Último recurso: devolver imágenes aleatorias
            print(f"  💥 Fallback extremo: imágenes aleatorias")
            random_indices = np.random.choice(len(df_inventory), k, replace=False)
            results = df_inventory.iloc[random_indices].copy()
            results['similarity_score'] = 0.1  # Score bajo
            method = "Emergencia (random)"
            pos_q_en = query_es
            return results, method, pos_q_en, False


def display_submission_results(results_df: pd.DataFrame, qid: str, query_es: str, query_en: str, method: str):
    """Muestra las imágenes de los resultados para una query específica."""
    
    # Crear título informativo
    title = f"Q{qid}: '{query_es}' → '{query_en}'\nMétodo: {method}"
    
    # Determinar columna de score
    score_col = 'reranked_score' if 'reranked_score' in results_df.columns else 'similarity_score'
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    fig.suptitle(title, fontsize=14, y=1.02)
    
    for i, (ax, (idx, row)) in enumerate(zip(axes, results_df.iterrows())):
        try:
            img_filename = row['file']
            img_path = os.path.join(images_path, img_filename)
            
            img = Image.open(img_path).convert('RGB')
            ax.imshow(img)
            
            # Mostrar score y posición
            score = row[score_col] if score_col in row else 0.0
            ax.set_title(f"#{i+1} Score: {score:.3f}", fontsize=10)
            ax.axis('off')
            
        except FileNotFoundError:
            ax.text(0.5, 0.5, f"[ERROR] No encontrado\n{img_filename}", 
                   ha='center', va='center', fontsize=8)
            ax.axis('off')
        except Exception as e:
            ax.text(0.5, 0.5, f"[ERROR] Error\n{str(e)[:30]}...", 
                   ha='center', va='center', fontsize=8)
            ax.axis('off')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    # Mostrar también en formato tabla
    print(f"  📊 Top {K} resultados (tabla):")
    display_cols = ['image_id', score_col, 'file']
    available_cols = [col for col in display_cols if col in results_df.columns]
    display(results_df[available_cols].head(K).style.format({
        score_col: "{:.3f}"
    }))


def display_text_only(results_df: pd.DataFrame, qid: str, query_es: str, query_en: str, method: str):
    """Muestra solo la información en texto para queries simples."""
    
    # Determinar columna de score
    score_col = 'reranked_score' if 'reranked_score' in results_df.columns else 'similarity_score'
    
    print(f"  📋 Resumen texto-only:")
    print(f"     Método: {method}")
    print(f"     Traducción: '{query_es}' → '{query_en}'")
    
    # Mostrar top 3 resultados en texto
    top_3 = results_df.head(3)
    print(f"     Top 3 resultados:")
    for i, (idx, row) in enumerate(top_3.iterrows()):
        score = row[score_col] if score_col in row else 0.0
        image_id = row['image_id'] if 'image_id' in row else row['file']
        print(f"       #{i+1}: {image_id} (score: {score:.3f})")
    
    # Mostrar tabla compacta
    print(f"  📊 Top {K} resultados (tabla compacta):")
    display_cols = ['image_id', score_col]
    available_cols = [col for col in display_cols if col in results_df.columns]
    display(results_df[available_cols].head(K).style.format({
        score_col: "{:.3f}"
    }).set_caption(f"Q{qid}: {query_es}"))

# --- DETECTAR QUERIES COMPLEJAS ---
def is_complex_query(qid: str, has_negation: bool, query_es: str) -> bool:
    """
    Determina si una query es compleja y merece visualización completa.
    """
    # Por número de query (20-40 son complejas)
    try:
        q_num = int(qid.replace('q', ''))
        if q_num >= 20:
            return True
    except:
        pass
    
    # Por contenido (contiene negación)
    if has_negation:
        return True
    
    # Por longitud o complejidad de la query
    complex_indicators = ['sin ', 'no ', 'excepto', 'pero no', 'pero sin', 'que no']
    if any(indicator in query_es.lower() for indicator in complex_indicators):
        return True
    
    return False

# --- Ejecutar el pipeline para todas las consultas CON VISUALIZACIÓN SELECTIVA ---
submission_data = []

if queries_es_list:
    print(f"\n🚀 Iniciando procesamiento de {len(queries_es_list)} consultas...")
    print("=" * 80)
    print("📝 NOTA: Visualización completa solo para queries complejas (20-40)")
    print("         Queries simples (1-19) mostrarán solo resumen en texto")
    print("=" * 80)
    
    for qid, query_es in tqdm(queries_es_list.items(), desc="Generando Submission"):
        print(f"\n{'='*50}")
        print(f"📋 Procesando Q{qid}: '{query_es}'")
        print(f"{'='*50}")
        
        try:
            # Ejecutar búsqueda segura con visualización
            results_df, method, query_en, has_negation = safe_search_with_display(query_es, qid, k=K)
            
            # VERIFICACIÓN CRÍTICA: Asegurar que tenemos la columna image_id
            if 'image_id' not in results_df.columns:
                print(f"  ⚠️ Creando image_id sobre la marcha para Q{qid}")
                results_df['image_id'] = results_df['file'].apply(
                    lambda x: os.path.splitext(x)[0]
                )
            
            # DECIDIR TIPO DE VISUALIZACIÓN
            is_complex = is_complex_query(qid, has_negation, query_es)
            
            if is_complex:
                print(f"  🎯 QUERY COMPLEJA - Mostrando visualización completa")
                display_submission_results(results_df, qid, query_es, query_en, method)
            else:
                print(f"  📄 QUERY SIMPLE - Mostrando resumen en texto")
                display_text_only(results_df, qid, query_es, query_en, method)
            
            # --- CÓDIGO CORREGIDO DENTRO DEL BUCLE ---

            # Asignar la columna de score final si no existe
            score_col = 'reranked_score' if 'reranked_score' in results_df.columns else 'similarity_score'
            if 'final_score' not in results_df.columns:
                 results_df['final_score'] = results_df[score_col]

            # 1. PASO CRÍTICO: ORDENAR POR EL SCORE FINAL (DESCENDENTE)
            # Esto garantiza que el más relevante (score más alto) esté primero.
            results_df.sort_values(by='final_score', ascending=False, inplace=True)
            # -----------------------------------------------------------------

            # 2. Formatear la columna 'preds' (top K resultados, ahora ordenados)
            top_k_images = results_df['image_id'].head(K).tolist()
            preds_string = ";".join(top_k_images)
            
            # ... (El resto del código para append a submission_data se mantiene igual)
            
            submission_data.append({
                "qid": qid,
                "preds": preds_string
            })
            
            print(f"  [SUCCESS] Q{qid} completado. Resultados: {len(top_k_images)} imágenes")
            print(f"  📝 Submission string: {preds_string[:100]}...")
            
        except Exception as e:
            print(f"  💥 ERROR CRÍTICO en Q{qid}: {e}")
            # Fallback de emergencia: imágenes aleatorias
            random_indices = np.random.choice(len(df_inventory), K, replace=False)
            random_images = df_inventory.iloc[random_indices]['file'].apply(
                lambda x: os.path.splitext(x)[0]
            ).tolist()
            preds_string = ";".join(random_images)
            
            submission_data.append({
                "qid": qid,
                "preds": preds_string
            })
            print(f"  🆘 Q{qid} usando fallback de emergencia")

    # --- Crear el archivo submission.csv ---
    df_submission = pd.DataFrame(submission_data)
    df_submission = df_submission[['qid', 'preds']]
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n{'='*80}")
    print(f"[SUCCESS] ARCHIVO DE SUBMISSION GENERADO EXITOSAMENTE")
    print(f"{'='*80}")
    print(f"📊 ESTADÍSTICAS FINALES:")
    print(f"   - Total de queries procesadas: {len(df_submission)}")
    print(f"   - Archivo guardado en: {SUBMISSION_PATH}")
    
    # Verificar que el archivo se creó correctamente
    if os.path.exists(SUBMISSION_PATH):
        file_size = os.path.getsize(SUBMISSION_PATH) / 1024  # KB
        print(f"   - Tamaño del archivo: {file_size:.2f} KB")
        
        # Mostrar preview del archivo de submission
        print(f"\n📋 PREVIEW DEL ARCHIVO SUBMISSION.CSV:")
        display(df_submission.head(10))
    else:
        print("   [ERROR] ERROR: No se pudo crear el archivo de submission")
        
else:
    print("No se pudieron cargar las consultas para procesar.")

print("\n🎯 ¡PROCESAMIENTO COMPLETADO!")